# Drowsiness eye-state model
This notebook documents the training pipeline used to produce `drowsiness_model.h5`. The recorded training run used a fixed 80/20 split with `random_state=42`, 10 epochs, and batch size 32. Re-running may differ by hardware and library versions.

## Dataset loading
The dataset is expected at `Drowsiness_Dataset/` with `Open_Eyes/` and `Close_Eyes/` folders. The class order is explicit: `0 = Open`, `1 = Closed`.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

DATASET = Path('Drowsiness_Dataset')
IMG_SIZE = 64
categories = ['Open_Eyes', 'Close_Eyes']
data, labels = [], []
for category in categories:
    for path in sorted((DATASET / category).glob('*.png')):
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise ValueError(f'Could not read {path}')
        data.append(cv2.resize(image, (IMG_SIZE, IMG_SIZE)))
        labels.append(categories.index(category))
data = np.asarray(data, dtype='float32') / 255.0
data = data.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
labels = to_categorical(np.asarray(labels), num_classes=2)
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42, stratify=labels)
print(len(data), len(X_train), len(X_test))

## Preprocessing
Every image is read as grayscale, resized to `64x64`, normalized by `/255`, and reshaped to `(64, 64, 1)`. Labels use one-hot encoding for the two softmax outputs.

## Architecture

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Input(shape=(64, 64, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Training
The original recorded run trained for 10 epochs with batch size 32 and used the held-out split as validation data.

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
model.save('drowsiness_model.h5')

## Evaluation
The original notebook recorded test accuracy `98.53%` and test loss `0.0412` for its run. Precision, recall, F1, and a confusion matrix are intentionally generated by `evaluate_model.py` instead of being guessed from accuracy.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_accuracy * 100:.2f}%')
print(f'Test loss: {test_loss:.4f}')